# 🚀 OpenMythos LoRA Fine-Tuning

**Free GPU Training on Google Colab (T4)**

This notebook demonstrates how to fine-tune OpenMythos models using LoRA (Low-Rank Adaptation) on free-tier GPUs. Supports QLoRA for even lower memory usage.

**What you'll learn:**
- Load an OpenMythos model (1B parameters)
- Apply LoRA adapters (train only ~0.5% of parameters)
- Fine-tune on custom finance/trading data
- Save and share your adapter

**Runtime:** T4 GPU (free tier)  
**Time:** ~30-60 minutes  
**Memory:** ~8GB VRAM with QLoRA

## 1. Setup & Installation

In [ ]:
# Install dependencies
!pip install torch transformers datasets accelerate

# Clone OpenMythos (our fork with LoRA support)
!git clone https://github.com/oyi77/OpenMythos.git
%cd OpenMythos
!pip install -e .

In [ ]:
# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print(f"PyTorch: {torch.__version__}")

## 2. Load Model

In [ ]:
from open_mythos import OpenMythos, mythos_1b
from open_mythos.lora import LoRAConfig, apply_lora, print_lora_summary
from open_mythos.quantization import quantize_model

# Create model
print("Loading mythos_1b...")
cfg = mythos_1b()
model = OpenMythos(cfg)

print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Apply QLoRA (quantized LoRA) for lower memory
# This reduces VRAM usage from ~16GB to ~8GB
USE_QLORA = True  # Set to False if you have 16GB+ VRAM

if USE_QLORA:
    print("Applying INT4 quantization...")
    model = quantize_model(model, bits=4, group_size=128)
    print("Quantization complete!")

In [ ]:
# Apply LoRA adapters
lora_config = LoRAConfig(
    rank=16,           # Low-rank dimension
    alpha=32,          # Scaling factor
    dropout=0.05,      # Dropout probability
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
)

model = apply_lora(model, lora_config)
print_lora_summary(model)

## 3. Prepare Dataset

In [ ]:
# Finance/trading training data
# Replace this with your own data!
training_data = [
    "Analyze XAUUSD: Gold trading at $2,350, resistance $2,380, support $2,320. RSI overbought on 4H. Wait for pullback to support for long entry.",
    "Business Plan: E-Commerce for Indonesian SMEs. Revenue: 2.5% transaction fee + IDR 50K/month premium. Target: 64M MSMEs. Year 1 projection: IDR 2.4B.",
    "Meta Ads Optimization: CPM dropped from IDR 15K to IDR 8.5K with Advantage+ audience. CTR: 1.2% → 2.8%. ROAS: 4.2x. Scale budget 50%.",
    "Cashflow: Revenue IDR 850M/month, expenses IDR 720M, profit IDR 130M. Burn rate: 3 months runway. Cut costs 15%, focus organic marketing.",
    "Trading Journal: LONG EUR/USD 1.0850, SL 1.0820, TP 1.0920. Risk 1%. Bullish engulfing daily, MACD crossover, USD weakness.",
    "Portfolio Review: 60% stocks, 20% crypto, 15% bonds, 5% cash. Rebalance: reduce crypto to 15%, increase bonds to 20%. Risk-adjusted return: 12.5%.",
    "SEO Strategy: Target long-tail keywords for Indonesian market. Content clusters: trading strategies, business plans, digital marketing. Expected traffic: 50K/month in 6 months.",
    "Affiliate Marketing: Shopee affiliate program. Commission: 5-10% per sale. Strategy: Product reviews + comparison content. Target: IDR 10M/month passive income.",
]

# Save to file
import json
with open("train_data.jsonl", "w") as f:
    for text in training_data:
        f.write(json.dumps({"text": text}) + "\n")

print(f"Created dataset with {len(training_data)} examples")

## 4. Training

In [ ]:
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
from torch.optim import AdamW
import time

# Simple dataset class
class TextDataset(Dataset):
    def __init__(self, path, tokenizer, max_length=512):
        self.examples = []
        with open(path) as f:
            for line in f:
                item = json.loads(line)
                self.examples.append(item["text"])
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        text = self.examples[idx]
        enc = self.tokenizer(text, max_length=self.max_length, 
                            truncation=True, padding="max_length",
                            return_tensors="pt")
        ids = enc["input_ids"].squeeze()
        return {"input_ids": ids, "labels": ids.clone()}

# Setup tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Create dataset and dataloader
dataset = TextDataset("train_data.jsonl", tokenizer, max_length=512)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

print(f"Dataset: {len(dataset)} examples")
print(f"Batch size: 2")

In [ ]:
# Training loop
model = model.cuda()

# Only train LoRA parameters
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=2e-4, weight_decay=0.01)

# Training
num_epochs = 3
print(f"\nStarting training for {num_epochs} epochs...")
print(f"Trainable parameters: {sum(p.numel() for p in trainable_params):,}")

for epoch in range(num_epochs):
    epoch_loss = 0
    start_time = time.time()
    
    for batch in dataloader:
        input_ids = batch["input_ids"].cuda()
        labels = batch["labels"].cuda()
        
        # Forward
        output = model(input_ids, labels=labels)
        loss = output.loss
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    elapsed = time.time() - start_time
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f} | Time: {elapsed:.1f}s")

print("\nTraining complete!")

## 5. Save & Share Adapter

In [ ]:
from open_mythos.lora import save_lora_adapter

# Save adapter (only ~1-10MB instead of full model)
save_lora_adapter(model, "my_finance_adapter.pt", config=lora_config)

# Check file size
import os
size_mb = os.path.getsize("my_finance_adapter.pt") / 1024 / 1024
print(f"Adapter saved: {size_mb:.1f} MB")
print("\nYou can now share this adapter file!")
print("Others can load it with: load_lora_adapter(model, 'my_finance_adapter.pt')")

## 6. Test Inference

In [ ]:
# Test the fine-tuned model
model.eval()

test_prompt = "Analyze XAUUSD trading opportunity:"
input_ids = tokenizer(test_prompt, return_tensors="pt").input_ids.cuda()

with torch.no_grad():
    output = model.generate(input_ids, max_new_tokens=100, n_loops=4)

generated = tokenizer.decode(output[0], skip_special_tokens=True)
print(f"Prompt: {test_prompt}")
print(f"\nGenerated:\n{generated}")

## 7. Next Steps

### What to do next:
1. **Replace training data** with your own dataset
2. **Increase epochs** for better results (10-20 epochs)
3. **Try different LoRA ranks** (8, 16, 32, 64)
4. **Upload adapter to HuggingFace** for community sharing

### Resources:
- [OpenMythos GitHub](https://github.com/oyi77/OpenMythos)
- [LoRA Paper](https://arxiv.org/abs/2106.09685)
- [QLoRA Paper](https://arxiv.org/abs/2305.14314)

### Community:
- Share your adapters on HuggingFace
- Open PRs with improvements
- Join the discussion on GitHub